# GTEx global alignment

Two analyses per GTEx tissue, each testing whether the selected SHAP LV(s) recover the true tissue by ORA against the GTEx tissue gene-set database:

1. **top1**: the single highest-ranked SHAP LV per tissue.
2. **cumulative25**: the top-ranked LVs needed to reach `CUMULATIVE_PCT`% of cumulative SHAP per tissue, the same criterion used in `00_LV_importance_kmeans.ipynb` / `01_LV_importance_kmeans_biology.ipynb`.

In [1]:
library(here)
library(dplyr)
library(tidyr)
library(stringr)
library(Matrix)
library(clusterProfiler)


SHAP_DIR <- here('output', '03_model_biology', '01_gtex',
                 '02_rf_kmeans', '00_LV_importance_kmeans',
                 'gtex_feature_importance_kmeans_binary_shap')
OUT_DIR  <- here('output', '03_model_biology', '01_gtex', '02_rf_kmeans', '06_global_alignment')
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

CLAMP_RDS      <- here('output', '01_model_building', '02_gtex', '01_CLAMP', 'CLAMPfull.rds')
GTEX_TISSUE_DB <- here('data', 'archs4', 'GTEx_Tissues_pathMat.rds')

N_LVS_PER_TISSUE <- 1L
CUMULATIVE_PCT   <- 25  # same threshold as 01_LV_importance_kmeans_biology.ipynb
TOP_GENE_PCT     <- 0.01
FDR_THRESH       <- 0.05

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses




Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union





Attaching package: ‘Matrix’




The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack




clusterProfiler v4.14.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

G Yu. Thirteen years of clusterProfiler. The Innovation. 2024,
5(6):100722




Attaching package: ‘clusterProfiler’




The following object is masked from ‘package:stats’:

    filter




In [2]:
shap_all <- read.delim(file.path(SHAP_DIR, 'all_shap_positive.tsv'),
                       stringsAsFactors = FALSE, check.names = FALSE)

selected_lvs_top1 <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::slice_head(n = N_LVS_PER_TISSUE) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selected_lvs_cum <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::mutate(
        reaches_thresh = Cumulative_Percent >= CUMULATIVE_PCT,
        cutoff_rank = if (any(reaches_thresh)) min(Rank[reaches_thresh]) else max(Rank)
    ) %>%
    dplyr::filter(Rank <= cutoff_rank) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selections <- list(top1 = selected_lvs_top1, cumulative25 = selected_lvs_cum)

cat('Tissues:', dplyr::n_distinct(selected_lvs_top1$Tissue), '\n')
for (nm in names(selections)) {
    cat(sprintf('  [%s] selected LV/tissue rows: %d, unique LVs: %d\n',
                nm, nrow(selections[[nm]]), dplyr::n_distinct(selections[[nm]]$LV)))
}

dplyr::bind_rows(lapply(names(selections), function(nm) {
    selections[[nm]] %>%
        dplyr::count(Tissue, name = 'n_selected_lvs') %>%
        dplyr::mutate(analysis = nm)
})) %>%
    tidyr::pivot_wider(names_from = analysis, values_from = n_selected_lvs) %>%
    dplyr::arrange(Tissue)

Tissues: 23 


  [top1] selected LV/tissue rows: 23, unique LVs: 21
  [cumulative25] selected LV/tissue rows: 79, unique LVs: 69


Tissue,top1,cumulative25
<chr>,<int>,<int>
Adrenal Gland,1,2
Artery - Tibial,1,4
Cells - Cultured fibroblasts,1,4
Cells - EBV-transformed lymphocytes,1,6
Colon - Transverse,1,3
Esophagus - Mucosa,1,3
Heart - Atrial Appendage,1,2
Heart - Left Ventricle,1,6
Kidney - Cortex,1,2


In [3]:
clamp <- readRDS(CLAMP_RDS)
Z_full <- as.matrix(clamp$Z)
rm(clamp)

all_selected_lvs <- unique(unlist(lapply(selections, function(df) df$LV)))
selected_unique_lvs <- intersect(all_selected_lvs, colnames(Z_full))
missing_lvs <- setdiff(all_selected_lvs, colnames(Z_full))
if (length(missing_lvs) > 0) warning('Missing LVs in Z: ', paste(missing_lvs, collapse = ', '))

Z <- Z_full[, selected_unique_lvs, drop = FALSE]
rm(Z_full)
universe_genes <- rownames(Z)
n_top_genes <- max(1L, ceiling(TOP_GENE_PCT * nrow(Z)))

cat('Z:', nrow(Z), 'genes x', ncol(Z), 'unique LVs (union across analyses)\n')
cat('Top genes per LV:', n_top_genes, '\n')

Z: 21613 genes x 69 unique LVs (union across analyses)


Top genes per LV: 217 


In [4]:
gtex_mat <- readRDS(GTEX_TISSUE_DB)
term_names <- unname(colnames(gtex_mat))

parse_gtex_tissue <- function(x) {
    x <- sub('^GTEx_Tissues_', '', x)
    x <- sub(' (Male|Female) [0-9]+-[0-9]+ Up$', '', x)
    x
}

normalize_text <- function(x) {
    x <- tolower(x)
    x <- gsub('[^a-z0-9]+', ' ', x)
    stringr::str_squish(x)
}

term2gene <- lapply(seq_along(term_names), function(i) {
    rownames(gtex_mat)[gtex_mat[, i] != 0]
})
names(term2gene) <- term_names

term2gene_df <- do.call(rbind, lapply(names(term2gene), function(term) {
    data.frame(term = term, gene = term2gene[[term]], stringsAsFactors = FALSE)
}))

term_map <- data.frame(
    term = term_names,
    parsed_tissue = parse_gtex_tissue(term_names),
    normalized_tissue = normalize_text(parse_gtex_tissue(term_names)),
    stringsAsFactors = FALSE
)

tissue_check <- dplyr::bind_rows(selections) %>%
    dplyr::distinct(Tissue) %>%
    dplyr::mutate(
        normalized_tissue = normalize_text(Tissue),
        matched_in_gtex_db = normalized_tissue %in% term_map$normalized_tissue,
        n_db_sets = vapply(normalized_tissue, function(x) sum(term_map$normalized_tissue == x), integer(1))
    )

cat('GTEx tissue gene sets:', length(term2gene), '\n')
stopifnot(all(tissue_check$matched_in_gtex_db))
tissue_check

GTEx tissue gene sets: 511 


Tissue,normalized_tissue,matched_in_gtex_db,n_db_sets
<chr>,<chr>,<lgl>,<int>
Adrenal Gland,adrenal gland,TRUE,11
Artery - Tibial,artery tibial,TRUE,12
Cells - Cultured fibroblasts,cells cultured fibroblasts,TRUE,12
Cells - EBV-transformed lymphocytes,cells ebv transformed lymphocytes,TRUE,9
Colon - Transverse,colon transverse,TRUE,11
Esophagus - Mucosa,esophagus mucosa,TRUE,12
Heart - Atrial Appendage,heart atrial appendage,TRUE,12
Heart - Left Ventricle,heart left ventricle,TRUE,12
Kidney - Cortex,kidney cortex,TRUE,7


In [5]:
run_gtex_tissue_ora <- function(genes, universe) {
    res <- tryCatch(
        clusterProfiler::enricher(
            gene          = genes,
            universe      = universe,
            TERM2GENE     = term2gene_df,
            pAdjustMethod = 'BH',
            pvalueCutoff  = 1,
            qvalueCutoff  = 1,
            minGSSize     = 10,
            maxGSSize     = 500
        ),
        error = function(e) NULL
    )
    if (is.null(res)) return(NULL)
    df <- as.data.frame(res)
    if (nrow(df) == 0) return(NULL)
    df
}

top_genes_per_lv <- lapply(colnames(Z), function(lv) {
    vals <- Z[, lv]
    universe_genes[order(vals, decreasing = TRUE)[seq_len(n_top_genes)]]
})
names(top_genes_per_lv) <- colnames(Z)

ora_list <- lapply(names(top_genes_per_lv), function(lv) {
    df <- run_gtex_tissue_ora(top_genes_per_lv[[lv]], universe_genes)
    if (is.null(df)) return(NULL)
    df$LV <- lv
    df
})

ora_all <- do.call(rbind, Filter(Negate(is.null), ora_list))
if (is.null(ora_all)) {
    ora_all <- data.frame()
} else {
    rownames(ora_all) <- NULL
    ora_all <- ora_all %>%
        dplyr::left_join(term_map, by = c('ID' = 'term')) %>%
        dplyr::select(LV, ID, Description, parsed_tissue, normalized_tissue,
                      GeneRatio, BgRatio, pvalue, p.adjust, qvalue, geneID, Count)
}

write.csv(ora_all, file.path(OUT_DIR, 'gtex_tissue_ora_per_lv.csv'), row.names = FALSE)
cat('ORA rows:', nrow(ora_all), '\n')
head(ora_all)

ORA rows: 17905 


,LV,ID,Description,parsed_tissue,normalized_tissue,GeneRatio,BgRatio,pvalue,p.adjust,qvalue,geneID,Count
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<int>
1,LV183,GTEx_Tissues_Adrenal Gland Male 40-49 Up,GTEx_Tissues_Adrenal Gland Male 40-49 Up,Adrenal Gland,adrenal gland,52/137,92/6121,2.918975e-65,8.202318e-63,8.202318e-63,CYP11B1/CYP17A1/CYP21A2/AS3MT/FAM166B/DBH/AMHR2/STAR/FAM43B/CYB561A3/FDXR/SNTB1/AADAC/IFITM10/KLHL4/SERPINA5/GSTA3/RMDN2/CYP11A1/FDX1/AKAP7/ABCB1/KCNK3/RGN/SLC16A9/RDX/KCNN2/MGARP/AKR1B1/ADGRV1/ASB4/ABCC3/NR5A1/KCNJ5/FNDC4/GSTA1/PORCN/SLC8B1/ALAS1/CYB5B/MCOLN3/KCTD14/KLHDC8B/DEXI/SEMA6A/MRAP/GALM/SREBF1/EPHX1/DLK1/C4B/C2CD2,52
2,LV183,GTEx_Tissues_Adrenal Gland Male 50-59 Up,GTEx_Tissues_Adrenal Gland Male 50-59 Up,Adrenal Gland,adrenal gland,52/137,93/6121,6.529532e-65,9.173993e-63,9.173993e-63,CYP11B1/CYP17A1/CYP21A2/AS3MT/FAM166B/AMHR2/STAR/FAM43B/CYB561A3/FDXR/SNTB1/AADAC/IFITM10/KLHL4/SERPINA5/GSTA3/RMDN2/CYP11A1/FDX1/AKAP7/ABCB1/KCNK3/RGN/SLC16A9/RDX/KCNN2/MGARP/AKR1B1/ADGRV1/ASB4/ABCC3/NR5A1/KCNJ5/FNDC4/GSTA1/PORCN/SLC8B1/ALAS1/CYB5B/MCOLN3/KCTD14/DEXI/SEMA6A/MRAP/GALM/SREBF1/MRPL33/EPHX1/DLK1/C4B/ZSWIM5/C2CD2,52
3,LV183,GTEx_Tissues_Adrenal Gland Male 20-29 Up,GTEx_Tissues_Adrenal Gland Male 20-29 Up,Adrenal Gland,adrenal gland,51/137,91/6121,1.157211e-63,1.083921e-61,1.083921e-61,CYP11B1/CYP17A1/CYP21A2/AS3MT/FAM166B/AMHR2/STAR/FAM43B/CYB561A3/FDXR/SNTB1/AADAC/IFITM10/KLHL4/SERPINA5/GSTA3/RMDN2/CYP11A1/FDX1/AKAP7/ABCB1/KCNK3/RGN/SLC16A9/RDX/KCNN2/MGARP/AKR1B1/ADGRV1/ASB4/ABCC3/NR5A1/KCNJ5/FNDC4/PORCN/SLC8B1/ALAS1/CYB5B/KLHDC8B/SLC37A2/ZNF117/SEMA6A/MRAP/GALM/TXN2/SREBF1/MRPL33/EPHX1/C4B/ZSWIM5/C2CD2,51
4,LV183,GTEx_Tissues_Adrenal Gland Female 20-29 Up,GTEx_Tissues_Adrenal Gland Female 20-29 Up,Adrenal Gland,adrenal gland,50/137,91/6121,9.840988e-62,5.530635e-60,5.530635e-60,CYP11B1/CYP17A1/CYP21A2/AS3MT/FAM166B/AMHR2/STAR/CYB561A3/FDXR/SNTB1/AADAC/KLHL4/SERPINA5/GSTA3/RMDN2/CYP11A1/FDX1/AKAP7/ABCB1/KCNK3/RGN/SLC16A9/CYSLTR2/RDX/KCNN2/MGARP/AKR1B1/ADGRV1/ASB4/ABCC3/NR5A1/KCNJ5/GSTA1/PORCN/SLC8B1/ALAS1/CYB5B/ACSF2/KLHDC8B/DEXI/MRAP/GALM/SREBF1/MRPL33/SLC46A1/EPHX1/DLK1/ZSWIM5/C2CD2/INHA,50
5,LV183,GTEx_Tissues_Adrenal Gland Female 60-69 Up,GTEx_Tissues_Adrenal Gland Female 60-69 Up,Adrenal Gland,adrenal gland,50/137,91/6121,9.840988e-62,5.530635e-60,5.530635e-60,CYP11B1/CYP17A1/CYP21A2/AS3MT/FAM166B/DBH/AMHR2/STAR/TH/CYB561A3/FDXR/SNTB1/AADAC/IFITM10/KLHL4/SERPINA5/GSTA3/RMDN2/CYP11A1/FDX1/AKAP7/ABCB1/KCNK3/RGN/SLC16A9/RDX/KCNN2/MGARP/NXPH1/AKR1B1/ASB4/ABCC3/NR5A1/KCNJ5/GSTA1/PORCN/SLC8B1/ALAS1/CYB5B/MCOLN3/KLHDC8B/DEXI/SEMA6A/MRAP/GALM/SREBF1/EPHX1/DLK1/ZSWIM5/C2CD2,50
6,LV183,GTEx_Tissues_Adrenal Gland Male 60-69 Up,GTEx_Tissues_Adrenal Gland Male 60-69 Up,Adrenal Gland,adrenal gland,50/137,92/6121,2.125165e-61,9.952854e-60,9.952854e-60,CYP11B1/CYP17A1/CYP21A2/AS3MT/FAM166B/AMHR2/STAR/FAM43B/CYB561A3/FDXR/SNTB1/AADAC/IFITM10/KLHL4/GSTA3/RMDN2/CYP11A1/FDX1/AKAP7/ABCB1/KCNK3/RGN/SLC16A9/RDX/KCNN2/MGARP/NXPH1/AKR1B1/ADGRV1/ASB4/ABCC3/NR5A1/KCNJ5/FNDC4/GSTA1/PORCN/SLC8B1/ALAS1/CYB5B/MCOLN3/KLHDC8B/DEXI/SEMA6A/MRAP/GALM/SREBF1/EPHX1/DLK1/C4B/C2CD2,50


In [6]:
run_alignment_summary <- function(selected_lvs, ora_all, out_dir) {
    dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
    sig_ora <- ora_all %>% dplyr::filter(LV %in% selected_lvs$LV, p.adjust < FDR_THRESH)

    detail <- selected_lvs %>%
        dplyr::rowwise() %>%
        dplyr::mutate(
            normalized_true_tissue = normalize_text(Tissue),
            n_sig_terms = sum(sig_ora$LV == LV),
            n_true_terms = sum(sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue),
            tissue_correct = n_true_terms > 0,
            best_any_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            best_true_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            matched_terms = paste(sig_ora$ID[sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue], collapse = ' | ')
        ) %>%
        dplyr::ungroup()

    tissue_summary <- detail %>%
        dplyr::group_by(Tissue) %>%
        dplyr::summarise(
            n_selected_lvs = dplyr::n(),
            tissue_correct = any(tissue_correct),
            correct_lvs = paste(LV[tissue_correct], collapse = ';'),
            best_true_padj = if (all(is.na(best_true_padj))) NA_real_ else min(best_true_padj, na.rm = TRUE),
            .groups = 'drop'
        ) %>%
        dplyr::mutate(correct_score = as.integer(tissue_correct))

    final_pct_tissue_correct <- 100 * mean(tissue_summary$tissue_correct)
    final_summary <- data.frame(
        n_tissues = nrow(tissue_summary),
        n_tissues_correct = sum(tissue_summary$tissue_correct),
        pct_tissue_correct = final_pct_tissue_correct,
        stringsAsFactors = FALSE
    )

    write.csv(detail, file.path(out_dir, 'gtex_global_alignment_detail.csv'), row.names = FALSE)
    write.csv(tissue_summary, file.path(out_dir, 'gtex_global_alignment_summary.csv'), row.names = FALSE)
    write.csv(final_summary, file.path(out_dir, 'gtex_global_alignment_final_pct.csv'), row.names = FALSE)

    list(detail = detail, tissue_summary = tissue_summary, final_summary = final_summary)
}

results <- lapply(names(selections), function(nm) {
    run_alignment_summary(selections[[nm]], ora_all, file.path(OUT_DIR, nm))
})
names(results) <- names(selections)

dplyr::bind_rows(lapply(names(results), function(nm) {
    results[[nm]]$final_summary %>% dplyr::mutate(analysis = nm, .before = 1)
}))

analysis,n_tissues,n_tissues_correct,pct_tissue_correct
<chr>,<int>,<int>,<dbl>
top1,23,21,91.30435
cumulative25,23,23,100.00000


In [7]:
cat('--- top1 ---\n')
results$top1$tissue_summary %>% dplyr::arrange(Tissue)

cat('--- cumulative25 ---\n')
results$cumulative25$tissue_summary %>% dplyr::arrange(Tissue)

--- top1 ---


Tissue,n_selected_lvs,tissue_correct,correct_lvs,best_true_padj,correct_score
<chr>,<int>,<lgl>,<chr>,<dbl>,<int>
Adrenal Gland,1,TRUE,LV183,8.202318e-63,1
Artery - Tibial,1,TRUE,LV563,3.743852e-05,1
Cells - Cultured fibroblasts,1,TRUE,LV546,8.440805e-59,1
Cells - EBV-transformed lymphocytes,1,FALSE,,NA,0
Colon - Transverse,1,TRUE,LV562,1.194764e-43,1
Esophagus - Mucosa,1,TRUE,LV84,2.870356e-82,1
Heart - Atrial Appendage,1,TRUE,LV420,2.313830e-62,1
Heart - Left Ventricle,1,TRUE,LV505,1.936986e-78,1
Kidney - Cortex,1,TRUE,LV229,7.554957e-44,1


--- cumulative25 ---


Tissue,n_selected_lvs,tissue_correct,correct_lvs,best_true_padj,correct_score
<chr>,<int>,<lgl>,<chr>,<dbl>,<int>
Adrenal Gland,2,TRUE,LV183;LV3,8.202318e-63,1
Artery - Tibial,4,TRUE,LV563;LV90;LV387;LV37,7.034525e-35,1
Cells - Cultured fibroblasts,4,TRUE,LV546;LV91;LV363;LV111,8.440805e-59,1
Cells - EBV-transformed lymphocytes,6,TRUE,LV21;LV123;LV378;LV547;LV116;LV90,1.084564e-27,1
Colon - Transverse,3,TRUE,LV562;LV368;LV108,1.915969e-55,1
Esophagus - Mucosa,3,TRUE,LV84;LV330;LV81,2.870356e-82,1
Heart - Atrial Appendage,2,TRUE,LV420;LV509,2.313830e-62,1
Heart - Left Ventricle,6,TRUE,LV505;LV554;LV76;LV137;LV9;LV203,1.936986e-78,1
Kidney - Cortex,2,TRUE,LV229;LV50,2.475097e-72,1


In [8]:
detail_cols <- c('Tissue', 'LV', 'Rank', 'Mean_SHAP_Tissue', 'Cumulative_Percent',
                 'tissue_correct', 'n_sig_terms', 'n_true_terms',
                 'best_true_padj', 'matched_terms')

cat('--- top1 ---\n')
results$top1$detail %>% dplyr::select(dplyr::all_of(detail_cols)) %>% dplyr::arrange(Tissue, Rank)

cat('--- cumulative25 ---\n')
results$cumulative25$detail %>% dplyr::select(dplyr::all_of(detail_cols)) %>% dplyr::arrange(Tissue, Rank)

--- top1 ---


Tissue,LV,Rank,Mean_SHAP_Tissue,Cumulative_Percent,tissue_correct,n_sig_terms,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<int>,<int>,<dbl>,<chr>
Adrenal Gland,LV183,1,0.13753360,14.365823,TRUE,11,11,8.202318e-63,GTEx_Tissues_Adrenal Gland Male 40-49 Up | GTEx_Tissues_Adrenal Gland Male 50-59 Up | GTEx_Tissues_Adrenal Gland Male 20-29 Up | GTEx_Tissues_Adrenal Gland Female 20-29 Up | GTEx_Tissues_Adrenal Gland Female 60-69 Up | GTEx_Tissues_Adrenal Gland Male 60-69 Up | GTEx_Tissues_Adrenal Gland Female 40-49 Up | GTEx_Tissues_Adrenal Gland Male 30-39 Up | GTEx_Tissues_Adrenal Gland Female 30-39 Up | GTEx_Tissues_Adrenal Gland Female 50-59 Up | GTEx_Tissues_Adrenal Gland Male 70-79 Up
Artery - Tibial,LV563,1,0.08563937,9.142689,TRUE,14,11,3.743852e-05,GTEx_Tissues_Artery - Tibial Female 30-39 Up | GTEx_Tissues_Artery - Tibial Male 20-29 Up | GTEx_Tissues_Artery - Tibial Female 40-49 Up | GTEx_Tissues_Artery - Tibial Male 40-49 Up | GTEx_Tissues_Artery - Tibial Female 20-29 Up | GTEx_Tissues_Artery - Tibial Male 30-39 Up | GTEx_Tissues_Artery - Tibial Female 70-79 Up | GTEx_Tissues_Artery - Tibial Male 60-69 Up | GTEx_Tissues_Artery - Tibial Male 50-59 Up | GTEx_Tissues_Artery - Tibial Male 70-79 Up | GTEx_Tissues_Artery - Tibial Female 50-59 Up
Cells - Cultured fibroblasts,LV546,1,0.11603051,12.029419,TRUE,12,12,8.440805e-59,GTEx_Tissues_Cells - Cultured Fibroblasts Female 40-49 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Male 50-59 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Male 60-69 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Male 40-49 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Female 50-59 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Male 20-29 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Female 60-69 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Male 30-39 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Male 70-79 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Female 30-39 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Female 20-29 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Female 70-79 Up
Cells - EBV-transformed lymphocytes,LV21,1,0.06096849,6.129557,FALSE,14,0,NA,
Colon - Transverse,LV562,1,0.10610394,11.353268,TRUE,26,10,1.194764e-43,GTEx_Tissues_Colon - Transverse Female 20-29 Up | GTEx_Tissues_Colon - Transverse Female 50-59 Up | GTEx_Tissues_Colon - Transverse Male 40-49 Up | GTEx_Tissues_Colon - Transverse Male 30-39 Up | GTEx_Tissues_Colon - Transverse Male 20-29 Up | GTEx_Tissues_Colon - Transverse Female 30-39 Up | GTEx_Tissues_Colon - Transverse Male 50-59 Up | GTEx_Tissues_Colon - Transverse Female 40-49 Up | GTEx_Tissues_Colon - Transverse Female 60-69 Up | GTEx_Tissues_Colon - Transverse Male 60-69 Up
Esophagus - Mucosa,LV84,1,0.13713745,14.194683,TRUE,20,12,2.870356e-82,GTEx_Tissues_Esophagus - Mucosa Male 40-49 Up | GTEx_Tissues_Esophagus - Mucosa Male 30-39 Up | GTEx_Tissues_Esophagus - Mucosa Male 50-59 Up | GTEx_Tissues_Esophagus - Mucosa Female 40-49 Up | GTEx_Tissues_Esophagus - Mucosa Male 20-29 Up | GTEx_Tissues_Esophagus - Mucosa Female 50-59 Up | GTEx_Tissues_Esophagus - Mucosa Female 20-29 Up | GTEx_Tissues_Esophagus - Mucosa Female 30-39 Up | GTEx_Tissues_Esophagus - Mucosa Female 60-69 Up | GTEx_Tissues_Esophagus - Mucosa Male 60-69 Up | GTEx_Tissues_Esophagus - Mucosa Female 70-79 Up | GTEx_Tissues_Esophagus - Mucosa Male 70-79 Up
Heart - Atrial Appendage,LV420,1,0.16581111,17.256570,TRUE,36,12,2.313830e-62,GTEx_Tissues_Heart - Atrial Appendage Male 70-79 Up | GTEx_Tissues_Heart - Atrial Appendage Male 40-49 Up | GTEx_Tissues_Heart - Atrial Appendage Female 40-49 Up | GTEx_Tissues_Heart - Atrial Appendage Female 60-69 Up | GTEx_Tissues_Heart - Atrial Appendage Male 20-29 Up | GTEx_Tissues_Heart - Atrial Appendage Male 50-59 Up | GTEx_Tissues_Heart - Atrial Appendage Male 60-69 Up | GTEx_Tissues_Heart - Atrial Appendage Female 50-59 Up | GTEx_Tissues_Heart - Atrial Appendage Female 70-79 Up | GTEx_Tissues_Heart - Atrial Appendage Female 3

--- cumulative25 ---


Tissue,LV,Rank,Mean_SHAP_Tissue,Cumulative_Percent,tissue_correct,n_sig_terms,n_true_terms,best_true_padj,matched_terms
<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<int>,<int>,<dbl>,<chr>
Adrenal Gland,LV183,1,0.13753360,14.365823,TRUE,11,11,8.202318e-63,GTEx_Tissues_Adrenal Gland Male 40-49 Up | GTEx_Tissues_Adrenal Gland Male 50-59 Up | GTEx_Tissues_Adrenal Gland Male 20-29 Up | GTEx_Tissues_Adrenal Gland Female 20-29 Up | GTEx_Tissues_Adrenal Gland Female 60-69 Up | GTEx_Tissues_Adrenal Gland Male 60-69 Up | GTEx_Tissues_Adrenal Gland Female 40-49 Up | GTEx_Tissues_Adrenal Gland Male 30-39 Up | GTEx_Tissues_Adrenal Gland Female 30-39 Up | GTEx_Tissues_Adrenal Gland Female 50-59 Up | GTEx_Tissues_Adrenal Gland Male 70-79 Up
Adrenal Gland,LV3,2,0.13109711,28.059334,TRUE,17,11,7.146307e-56,GTEx_Tissues_Adrenal Gland Male 50-59 Up | GTEx_Tissues_Adrenal Gland Female 40-49 Up | GTEx_Tissues_Adrenal Gland Female 60-69 Up | GTEx_Tissues_Adrenal Gland Female 50-59 Up | GTEx_Tissues_Adrenal Gland Male 60-69 Up | GTEx_Tissues_Adrenal Gland Female 20-29 Up | GTEx_Tissues_Adrenal Gland Male 30-39 Up | GTEx_Tissues_Adrenal Gland Male 40-49 Up | GTEx_Tissues_Adrenal Gland Female 30-39 Up | GTEx_Tissues_Adrenal Gland Male 20-29 Up | GTEx_Tissues_Adrenal Gland Male 70-79 Up
Artery - Tibial,LV563,1,0.08563937,9.142689,TRUE,14,11,3.743852e-05,GTEx_Tissues_Artery - Tibial Female 30-39 Up | GTEx_Tissues_Artery - Tibial Male 20-29 Up | GTEx_Tissues_Artery - Tibial Female 40-49 Up | GTEx_Tissues_Artery - Tibial Male 40-49 Up | GTEx_Tissues_Artery - Tibial Female 20-29 Up | GTEx_Tissues_Artery - Tibial Male 30-39 Up | GTEx_Tissues_Artery - Tibial Female 70-79 Up | GTEx_Tissues_Artery - Tibial Male 60-69 Up | GTEx_Tissues_Artery - Tibial Male 50-59 Up | GTEx_Tissues_Artery - Tibial Male 70-79 Up | GTEx_Tissues_Artery - Tibial Female 50-59 Up
Artery - Tibial,LV90,2,0.07622058,17.279846,TRUE,35,12,3.205225e-04,GTEx_Tissues_Artery - Tibial Female 50-59 Up | GTEx_Tissues_Artery - Tibial Female 60-69 Up | GTEx_Tissues_Artery - Tibial Female 70-79 Up | GTEx_Tissues_Artery - Tibial Male 60-69 Up | GTEx_Tissues_Artery - Tibial Male 40-49 Up | GTEx_Tissues_Artery - Tibial Male 50-59 Up | GTEx_Tissues_Artery - Tibial Male 70-79 Up | GTEx_Tissues_Artery - Tibial Female 40-49 Up | GTEx_Tissues_Artery - Tibial Male 30-39 Up | GTEx_Tissues_Artery - Tibial Female 20-29 Up | GTEx_Tissues_Artery - Tibial Male 20-29 Up | GTEx_Tissues_Artery - Tibial Female 30-39 Up
Artery - Tibial,LV387,3,0.05017576,22.636511,TRUE,56,12,7.034525e-35,GTEx_Tissues_Artery - Tibial Female 50-59 Up | GTEx_Tissues_Artery - Tibial Female 60-69 Up | GTEx_Tissues_Artery - Tibial Male 60-69 Up | GTEx_Tissues_Artery - Tibial Male 50-59 Up | GTEx_Tissues_Artery - Tibial Male 30-39 Up | GTEx_Tissues_Artery - Tibial Female 20-29 Up | GTEx_Tissues_Artery - Tibial Female 70-79 Up | GTEx_Tissues_Artery - Tibial Male 40-49 Up | GTEx_Tissues_Artery - Tibial Male 70-79 Up | GTEx_Tissues_Artery - Tibial Female 40-49 Up | GTEx_Tissues_Artery - Tibial Female 30-39 Up | GTEx_Tissues_Artery - Tibial Male 20-29 Up
Artery - Tibial,LV37,4,0.04984620,27.957992,TRUE,36,12,1.358742e-18,GTEx_Tissues_Artery - Tibial Male 60-69 Up | GTEx_Tissues_Artery - Tibial Male 50-59 Up | GTEx_Tissues_Artery - Tibial Female 70-79 Up | GTEx_Tissues_Artery - Tibial Male 70-79 Up | GTEx_Tissues_Artery - Tibial Female 60-69 Up | GTEx_Tissues_Artery - Tibial Female 50-59 Up | GTEx_Tissues_Artery - Tibial Male 40-49 Up | GTEx_Tissues_Artery - Tibial Female 40-49 Up | GTEx_Tissues_Artery - Tibial Male 30-39 Up | GTEx_Tissues_Artery - Tibial Female 30-39 Up | GTEx_Tissues_Artery - Tibial Female 20-29 Up | GTEx_Tissues_Artery - Tibial Male 20-29 Up
Cells - Cultured fibroblasts,LV546,1,0.11603051,12.029419,TRUE,12,12,8.440805e-59,GTEx_Tissues_Cells - Cultured Fibroblasts Female 40-49 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Male 50-59 Up | GTEx_Tissues_Cells - Cultured Fibroblasts Male 60-69 Up | GTEx_Tissues_Cells - Cultured Fibroblasts 

In [9]:
# Per-LV correctness within the cumulative25 selection (not just "any LV correct")
per_lv_summary <- results$cumulative25$detail %>%
    dplyr::group_by(Tissue) %>%
    dplyr::summarise(
        n_lvs = dplyr::n(),
        n_correct = sum(tissue_correct),
        pct_correct = 100 * n_correct / n_lvs,
        .groups = "drop"
    ) %>%
    dplyr::arrange(pct_correct)

print(per_lv_summary, n = Inf)

total_lvs <- sum(per_lv_summary$n_lvs)
total_correct <- sum(per_lv_summary$n_correct)
cat(sprintf(
    "\nOverall: %d/%d LV-tissue rows correct (%.1f%%) across cumulative25 selection\n",
    total_correct, total_lvs, 100 * total_correct / total_lvs
))
cat(sprintf("Tissues at 100%% LV concordance: %d/%d\n",
            sum(per_lv_summary$pct_correct == 100), nrow(per_lv_summary)))

write.csv(per_lv_summary, file.path(OUT_DIR, "cumulative25", "gtex_global_alignment_per_lv_pct.csv"), row.names = FALSE)


# A tibble: 23 × 4
   Tissue                              n_lvs n_correct pct_correct
   <chr>                               <int>     <int>       <dbl>
 1 Cells - Cultured fibroblasts            4         1        25  
 2 Testis                                  4         1        25  
 3 Whole Blood                             4         1        25  
 4 Cells - EBV-transformed lymphocytes     6         2        33.3
 5 Heart - Left Ventricle                  6         2        33.3
 6 Muscle - Skeletal                       3         1        33.3
 7 Esophagus - Mucosa                      3         2        66.7
 8 Lung                                    3         2        66.7
 9 Nerve - Tibial                          3         2        66.7
10 Vagina                                  3         2        66.7
11 Pituitary                               5         4        80  
12 Small Intestine - Terminal Ileum        5         4        80  
13 Adrenal Gland                           


Overall: 54/79 LV-tissue rows correct (68.4%) across cumulative25 selection


Tissues at 100% LV concordance: 11/23


In [10]:
stopifnot(file.exists(file.path(OUT_DIR, 'gtex_tissue_ora_per_lv.csv')))

for (nm in names(selections)) {
    out_dir <- file.path(OUT_DIR, nm)
    expected_rows <- nrow(selections[[nm]])
    stopifnot(nrow(results[[nm]]$detail) == expected_rows)
    stopifnot(nrow(results[[nm]]$tissue_summary) == dplyr::n_distinct(shap_all$Tissue))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_detail.csv')))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_summary.csv')))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_final_pct.csv')))
    cat(sprintf(
        '[%s] Checks passed. Denominator = %d tissues and %d selected LV/tissue rows. Final %% tissue correct = %.2f\n',
        nm, nrow(results[[nm]]$tissue_summary), nrow(results[[nm]]$detail),
        results[[nm]]$final_summary$pct_tissue_correct
    ))
}

[top1] Checks passed. Denominator = 23 tissues and 23 selected LV/tissue rows. Final % tissue correct = 91.30
[cumulative25] Checks passed. Denominator = 23 tissues and 79 selected LV/tissue rows. Final % tissue correct = 100.00
